### Import packages

In [1]:
!pip install hanlp
!pip install transformers==4.46.3 --force-reinstall -q

  Using cached hanlp-2.1.3-py3-none-any.whl.metadata (13 kB)
Using cached hanlp-2.1.3-py3-none-any.whl (654 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
textdescriptives 2.8.4 requires numpy<2.0.0,>=1.20.0, but you have numpy 2.4.4 which is incompatible.
datasets 4.7.0 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.


In [21]:
import hanlp
import json
import csv
import re
from pathlib import Path
from tqdm import tqdm

In [14]:
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/cleaned"
GENDERED_NAMES_DIR = PROJECT_ROOT / "data/gendered_names"

In [8]:
HanLP = hanlp.load(hanlp.pretrained.mtl.UD_ONTONOTES_TOK_POS_LEM_FEA_NER_SRL_DEP_SDP_CON_MMINILMV2L6)

/Users/tildeidunsloth/Desktop/Thesis/Thesis_project/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Users/tildeidunsloth/Desktop/Thesis/Thesis_project/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
sci_fi_data_path = DATA_DIR / "sci_fi_stories_cleaned.jsonl"
romance_data_path = DATA_DIR / "romance_stories_cleaned.jsonl"
literary_fiction_data_path = DATA_DIR / "lit_fiction_stories_cleaned.jsonl"

In [26]:
def load_jsonl(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

In [27]:
lit_fic_stories = load_jsonl(literary_fiction_data_path)
romance_stories = load_jsonl(romance_data_path)
sci_fi_stories = load_jsonl(sci_fi_data_path)

In [28]:
# Load previously extracted gendered names
with open(GENDERED_NAMES_DIR / "male_names_sci_fi.json") as f:
    male_names_sci_fi = json.load(f)

with open(GENDERED_NAMES_DIR / "female_names_sci_fi.json") as f:
    female_names_sci_fi = json.load(f)

with open(GENDERED_NAMES_DIR / "male_names_lit.json") as f:
    male_names_lit = json.load(f)

with open(GENDERED_NAMES_DIR / "female_names_lit.json") as f:
    female_names_lit = json.load(f)

with open(GENDERED_NAMES_DIR / "male_names_romance.json") as f:
    male_names_romance = json.load(f)
    
with open(GENDERED_NAMES_DIR / "female_names_romance.json") as f:
    female_names_romance = json.load(f)

In [29]:
result = HanLP(["The chef cooked a delicious meal for the guests yesterday."], tasks='srl')
print(result)

{
  "tok": [
    ["The", "chef", "cooked", "a", "delicious", "meal", "for", "the", "guests", "yesterday", "."]
  ],
  "srl": [
    [[["The chef", "ARG0", 0, 2], ["cooked", "PRED", 2, 3], ["a delicious meal", "ARG1", 3, 6], ["for the guests", "ARG2", 6, 9], ["yesterday", "ARGM-TMP", 9, 10]]]
  ]
}


In [37]:
LIT_NAMES = female_names_lit + male_names_lit
ROMANCE_NAMES = female_names_romance + male_names_romance
SCI_FI_NAMES = female_names_lit + male_names_lit

In [41]:
# Gendered words
NAME_SET = set(name.lower() for name in LIT_NAMES)
GENDERED_PRONOUNS = {'he', 'him', 'his', 'himself', 'she', 'her', 'hers', 'herself'}
GENDERED_NOUNS = {'daughter', 'mother', 'woman', 'girl', 'female', 'sister', 'daughters', 'mothers', 'women', 'girls', 'females', 'sisters', 'aunt', 'aunts', 'niece', 'nieces', 'mrs.', 'miss',
'son', 'father', 'man', 'boy', 'male', 'brother', 'sons', 'fathers', 'men', 'boys', 'males', 'brothers', 'uncle', 'uncles', 'nephew', 'nephews', 'mr.'}

ROLES_OF_INTEREST = {"ARG0", "ARG1", "ARG2", "PRED"}

def split_long_sentence(sentence, max_length=280):
    chunks = []
    while len(sentence) > max_length:
        split_idx = sentence[:max_length].rfind(' ')
        if split_idx == -1:
            split_idx = max_length
        chunks.append(sentence[:split_idx].strip())
        sentence = sentence[split_idx:].strip()
    if sentence:
        chunks.append(sentence)
    return chunks

def contains_candidate_token(sentence):
    tokens = re.findall(r'\b\w+\b', sentence.lower())
    return bool(set(tokens) & (GENDERED_PRONOUNS | GENDERED_NOUNS | NAME_SET))

def is_relevant_span(span):
    tokens = re.findall(r'\b\w+\b', span.lower())
    return bool(set(tokens) & (GENDERED_PRONOUNS | GENDERED_NOUNS | NAME_SET))

def process_stories_to_csv(stories, genre, output_path, HanLP, mode='w', max_chunk_length=280):
    """Predicate-centered CSV keeping pronouns, gendered nouns, and first names (NER)."""
    fieldnames = ["story_id", "predicate", "agent", "patient", "receiver", "genre",
                  "agent_type", "event_valence", "context_valence"]

    with open(output_path, mode, newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if mode == 'w':
            writer.writeheader()

        for story_idx, story in enumerate(tqdm(stories, desc=f"Processing {genre}", unit="story")):
            story_id = f"{genre.lower().replace(' ', '_')}_{story_idx + 1}"
            text = story.get("sentences", story.get("story", ""))
            if isinstance(text, list):
                text = " ".join(text)

            agent_type = story.get("agent_type", None)
            event_valence = story.get("event_valence", None)
            context_valence = story.get("context_valence", None)

            sentences = re.split(r'(?<=[.!?])\s+', text)

            for sentence in sentences:
                if not sentence.strip():
                    continue

                for chunk in split_long_sentence(sentence, max_length=max_chunk_length):
                    if not contains_candidate_token(chunk):
                        continue  # Skip chunks with no relevant content

                    try:
                        # Run SRL
                        result = HanLP([chunk], tasks=['srl'])
                        srl_frames = result['srl'][0]

                        for frame in srl_frames:
                            predicate = None
                            agent = None
                            patient = None
                            receiver = None

                            for span, role, start, end in frame:
                                if role == "PRED":
                                    predicate = span
                                elif role == "ARG0" and is_relevant_span(span):
                                    agent = span
                                elif role == "ARG1" and is_relevant_span(span):
                                    patient = span
                                elif role == "ARG2" and is_relevant_span(span):
                                    receiver = span

                            # Only save if predicate exists and at least one relevant role
                            if predicate and (agent or patient or receiver):
                                writer.writerow({
                                    "story_id": story_id,
                                    "predicate": predicate,
                                    "agent": agent,
                                    "patient": patient,
                                    "receiver": receiver,
                                    "genre": genre,
                                    "agent_type": agent_type,
                                    "event_valence": event_valence,
                                    "context_valence": context_valence
                                })

                    except Exception as e:
                        print(f"Error processing story {story_id}: {e}")
                        continue

    print(f"{genre} stories processed and saved to {output_path}")

In [42]:
process_stories_to_csv(lit_fic_stories, "Literary_fiction", "literary_fiction_srl.csv", HanLP)

Processing Literary_fiction: 100%|██████████| 5800/5800 [1:53:23<00:00,  1.17s/story]  

Literary_fiction stories processed and saved to literary_fiction_srl.csv
